# 9.1 最终评估

把 3.1 封存至今的最终评估集拿出来，对 7.1 的四个候选**只评一次**：换一批没参与过拟合和筛选的行，理化指标带来的改善还在不在。
模型原样加载，不重训、不改参数；结果无论好坏都不回头改 7.1。


In [1]:
import hashlib
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import dsflow

ROOT = Path.cwd()
while not (ROOT / "dsflow.yaml").is_file():
    ROOT = ROOT.parent
STEP = ROOT / "steps/09_评估与诊断/9.1_最终评估"
OUT = STEP / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
S31 = ROOT / "steps/03_数据划分/3.1_预测任务定义_固定划分与验证基线/outputs"
S71 = ROOT / "steps/07_模型选择与训练/7.1_候选模型比较/outputs"
FEATURES = ["fixed acidity", "volatile acidity", "citric acid", "residual sugar", "chlorides",
            "free sulfur dioxide", "total sulfur dioxide", "density", "pH", "sulphates", "alcohol"]
WINES = {"red": "红酒", "white": "白酒"}
CANDIDATES = ("Ridge", "随机森林")
SEED = 20260914
DRAWS = 2000

run = dsflow.start_run("9.1", project=ROOT, hypothesis="7.1 在验证集上筛出的四个候选，在封存的最终评估集上仍比基线的 MAE 低")
raw = {}
for w, name in WINES.items():
    path = ROOT / f"data/winequality-{w}.csv"
    run.log_input(path, name=name)
    raw[w] = pd.read_csv(path, sep=";").assign(source_row=lambda d: np.arange(1, len(d) + 1))
run.log_input(S31 / "split_assignments.csv", name="划分表")
splits = pd.read_csv(S31 / "split_assignments.csv")
comparison = json.loads((S71 / "model_comparison.json").read_text(encoding="utf-8"))
models = {(w, c): joblib.load(S71 / "models" / f"{w}_{c}.joblib") for w in WINES for c in CANDIDATES}
INPUT_FILES = ["data/winequality-red.csv", "data/winequality-white.csv",
               "steps/03_数据划分/3.1_预测任务定义_固定划分与验证基线/outputs/split_assignments.csv",
               "steps/03_数据划分/3.1_预测任务定义_固定划分与验证基线/outputs/baseline_validation.json",
               "steps/07_模型选择与训练/7.1_候选模型比较/outputs/model_comparison.json",
               *[f"steps/07_模型选择与训练/7.1_候选模型比较/outputs/models/{w}_{c}.joblib" for w in WINES for c in CANDIDATES],
               *[f"steps/07_模型选择与训练/7.1_候选模型比较/outputs/scaler_{w}.json" for w in WINES]]
fingerprint = lambda rel: {"bytes": (ROOT / rel).stat().st_size, "sha256": hashlib.sha256((ROOT / rel).read_bytes()).hexdigest()}
inputs_before = {rel: fingerprint(rel) for rel in INPUT_FILES}
run.log_params({"模型": "7.1 的四个候选原样加载，不重训", "评估次数": "每个候选一次", "重抽样": {"次数": DRAWS, "种子": SEED, "生成器": "PCG64"},
                "模型文件哈希": {f"{w}_{c}": inputs_before[f"steps/07_模型选择与训练/7.1_候选模型比较/outputs/models/{w}_{c}.joblib"]["sha256"][:12] for w in WINES for c in CANDIDATES}})
print(f"加载 {len(models)} 个候选模型；红酒 {len(raw['red']):,} 行、白酒 {len(raw['white']):,} 行；划分表 {len(splits):,} 行")


加载 4 个候选模型；红酒 1,599 行、白酒 4,898 行；划分表 6,497 行


In [2]:
pred_rows, summary, data = [], {}, {}
for w, name in WINES.items():
    df = raw[w].merge(splits[splits["wine"] == w].drop(columns="wine"), on="source_row", how="left", validate="one_to_one")
    assert df["split"].notna().all()
    final = df[df["split"] == "final_evaluation"].sort_values("source_row").reset_index(drop=True)
    n_train, n_valid = int((df["split"] == "training").sum()), int((df["split"] == "validation").sum())
    median = comparison["files"][w]["baseline"]["prediction"]
    y = final["quality"].to_numpy(float)
    X = final[FEATURES].to_numpy(float)
    base_err = np.abs(y - median)
    base = {"prediction": median, "mae": float(base_err.mean()), "rmse": float(np.sqrt((base_err ** 2).mean()))}
    summary[w] = {"name": name, "final_rows": len(final), "final_feature_groups": int(final["feature_group_id"].nunique()),
                  "training_rows_untouched": n_train, "validation_rows_untouched": n_valid, "baseline": base, "candidates": {}}
    data[w] = final
    for cand in CANDIDATES:
        m = models[(w, cand)]
        p = m["model"].predict((X - m["scaler"]["mean"]) / m["scaler"]["std"]) if "scaler" in m else m["model"].predict(X)
        assert np.isfinite(p).all()
        err = np.abs(y - p)
        v71 = comparison["files"][w]["candidates"][cand]
        summary[w]["candidates"][cand] = {"mae": float(err.mean()), "rmse": float(np.sqrt((err ** 2).mean())),
                                          "mae_minus_baseline": float(err.mean() - base["mae"]), "rmse_minus_baseline": float(np.sqrt((err ** 2).mean()) - base["rmse"]),
                                          "validation_mae_from_7_1": v71["mae"], "validation_mae_minus_baseline_from_7_1": v71["mae_minus_baseline"]}
        run.log_metrics({f"{name}_{cand}_MAE_最终评估": float(err.mean()), f"{name}_{cand}_RMSE_最终评估": float(np.sqrt((err ** 2).mean()))})
        for i in range(len(y)):
            pred_rows.append({"wine": w, "source_row": int(final["source_row"][i]), "feature_group_id": final["feature_group_id"][i], "candidate": cand,
                              "quality": float(y[i]), "prediction": float(p[i]), "baseline_prediction": median,
                              "abs_error": float(err[i]), "baseline_abs_error": float(base_err[i])})
    run.log_metrics({f"{name}_基线_MAE_最终评估": base["mae"], f"{name}_基线_RMSE_最终评估": base["rmse"]})
predictions = pd.DataFrame(pred_rows)
predictions.to_csv(OUT / "final_predictions.csv", index=False)
table = pd.DataFrame([{"酒类": s["name"], "候选": c, "最终评估 MAE": round(v["mae"], 4), "基线 MAE": round(s["baseline"]["mae"], 4),
                       "MAE 差": round(v["mae_minus_baseline"], 4), "验证集 MAE（7.1）": round(v["validation_mae_from_7_1"], 4),
                       "最终评估 RMSE": round(v["rmse"], 4), "基线 RMSE": round(s["baseline"]["rmse"], 4)}
                      for s in summary.values() for c, v in s["candidates"].items()])
print("最终评估行：红酒 %d、白酒 %d；训练与验证行一行没用" % (summary["red"]["final_rows"], summary["white"]["final_rows"]))
print(table.to_string(index=False))


最终评估行：红酒 318、白酒 994；训练与验证行一行没用
酒类    候选  最终评估 MAE  基线 MAE   MAE 差  验证集 MAE（7.1）  最终评估 RMSE  基线 RMSE
红酒 Ridge    0.5090  0.6855 -0.1766        0.5194     0.6548   0.9042
红酒  随机森林    0.4963  0.6855 -0.1892        0.5051     0.6549   0.9042
白酒 Ridge    0.6194  0.6469 -0.0275        0.5814     0.7899   0.9121
白酒  随机森林    0.5752  0.6469 -0.0717        0.5474     0.7278   0.9121


In [3]:
draws, diff_rows = {}, []
for w, name in WINES.items():
    final = data[w]
    ids = sorted(final["feature_group_id"].unique())
    G = len(ids)
    pos = {g: i for i, g in enumerate(ids)}
    grp = final["feature_group_id"].map(pos).to_numpy()
    rows_per_group = np.bincount(grp, minlength=G)
    median = summary[w]["baseline"]["prediction"]
    err_sum = {"基线": np.bincount(grp, weights=np.abs(final["quality"].to_numpy(float) - median), minlength=G)}
    for cand in CANDIDATES:
        p = predictions[(predictions["wine"] == w) & (predictions["candidate"] == cand)].set_index("source_row").loc[final["source_row"]]
        err_sum[cand] = np.bincount(grp, weights=p["abs_error"].to_numpy(), minlength=G)
    rng = np.random.Generator(np.random.PCG64(SEED))
    idx = rng.integers(0, G, size=(DRAWS, G))
    draws[w] = {"ordered_feature_group_ids": ids, "indices": idx.tolist()}
    for cand in CANDIDATES:
        diffs = np.empty(DRAWS)
        for b in range(DRAWS):
            times = np.bincount(idx[b], minlength=G)
            n = float((times * rows_per_group).sum())
            diffs[b] = (times * err_sum[cand]).sum() / n - (times * err_sum["基线"]).sum() / n
        lo, hi = (float(x) for x in np.percentile(diffs, [2.5, 97.5], method="linear"))
        summary[w]["candidates"][cand]["mae_difference_interval"] = {"lower_2_5": lo, "upper_97_5": hi, "draws": DRAWS}
        run.log_metrics({f"{name}_{cand}_MAE差_最终评估_区间上界": hi})
        diff_rows += [{"draw": b, "wine": w, "candidate": cand, "mae_minus_baseline": float(diffs[b])} for b in range(DRAWS)]
        print(f"{name} {cand}：最终评估集上「候选 MAE − 基线 MAE」的 2.5%～97.5% 区间 [{lo:.4f}, {hi:.4f}]（{G} 个特征组合）")
(OUT / "bootstrap_draws.json").write_text(json.dumps({"seed": SEED, "generator": "numpy.random.Generator(numpy.random.PCG64(seed))", "draws": DRAWS,
    "rule": "同 7.1：每次从 G 个最终评估特征组合等概率有放回抽 G 个索引；抽中几次其全部行计几次；按抽中后的原始行数求 MAE；候选与基线共用索引",
    "wines": draws}, ensure_ascii=False), encoding="utf-8")
pd.DataFrame(diff_rows).to_csv(OUT / "bootstrap_mae_differences.csv", index=False)


红酒 Ridge：最终评估集上「候选 MAE − 基线 MAE」的 2.5%～97.5% 区间 [-0.2490, -0.1008]（274 个特征组合）
红酒 随机森林：最终评估集上「候选 MAE − 基线 MAE」的 2.5%～97.5% 区间 [-0.2642, -0.1145]（274 个特征组合）
白酒 Ridge：最终评估集上「候选 MAE − 基线 MAE」的 2.5%～97.5% 区间 [-0.0637, 0.0107]（794 个特征组合）
白酒 随机森林：最终评估集上「候选 MAE − 基线 MAE」的 2.5%～97.5% 区间 [-0.1120, -0.0313]（794 个特征组合）


In [4]:
for w, name in WINES.items():
    final = data[w]
    by_q = {}
    for q, g in final.groupby("quality"):
        entry = {"rows": int(len(g)), "feature_groups": int(g["feature_group_id"].nunique()),
                 "baseline_mae": float(np.abs(g["quality"] - summary[w]["baseline"]["prediction"]).mean())}
        for cand in CANDIDATES:
            p = predictions[(predictions["wine"] == w) & (predictions["candidate"] == cand) & predictions["source_row"].isin(g["source_row"])]
            entry[f"{cand}_mae"] = float(p["abs_error"].mean())
        by_q[str(int(q))] = entry
    summary[w]["by_quality"] = by_q
    for cand, v in summary[w]["candidates"].items():
        v["still_better_than_baseline"] = bool(v["mae"] < summary[w]["baseline"]["mae"] and v["mae_difference_interval"]["upper_97_5"] < 0)
        v["final_minus_validation_mae"] = v["mae"] - v["validation_mae_from_7_1"]
inputs_after = {rel: fingerprint(rel) for rel in INPUT_FILES}
assert inputs_after == inputs_before
result = {"note": "每个候选在最终评估集上只评了这一次；最终评估行没有参与过任何拟合、预测或筛选，但在 2.1 的全量探索里被看过分布与相关",
          "features": FEATURES, "inputs_before": inputs_before, "inputs_after": inputs_after, "inputs_unchanged": True, "files": summary}
(OUT / "final_evaluation.json").write_text(json.dumps(result, ensure_ascii=False, indent=1), encoding="utf-8")
run.log_artifact(OUT / "final_evaluation.json", purpose="每类酒每候选在最终评估集上的 MAE / RMSE、相对基线的差、区间、按分值误差，并列 7.1 的验证集 MAE", kind="table")
run.log_artifact(OUT / "final_predictions.csv", purpose="每个候选对每条最终评估行的唯一一次预测", kind="table")
run.log_artifact(OUT / "bootstrap_draws.json", purpose="最终评估集重抽样的有序组合 ID 与全部抽样索引", kind="other")
run.log_artifact(OUT / "bootstrap_mae_differences.csv", purpose="最终评估集重抽样每次每候选的 MAE 配对差", kind="table")
best = {w: min(CANDIDATES, key=lambda c: summary[w]["candidates"][c]["mae"]) for w in WINES}
for w, name in WINES.items():
    for cand, v in summary[w]["candidates"].items():
        print(f"{name} {cand}：最终评估 MAE {v['mae']:.4f}，验证集 {v['validation_mae_from_7_1']:.4f}，差 {v['final_minus_validation_mae']:+.4f}；"
              f"仍优于基线：{'是' if v['still_better_than_baseline'] else '否'}")
print(pd.DataFrame([{"酒类": WINES[w], "quality": q, **e} for w in WINES for q, e in summary[w]["by_quality"].items()]).round(4).to_string(index=False))
print("每类酒最终评估 MAE 最低的候选：", {WINES[w]: c for w, c in best.items()})


红酒 Ridge：最终评估 MAE 0.5090，验证集 0.5194，差 -0.0104；仍优于基线：是
红酒 随机森林：最终评估 MAE 0.4963，验证集 0.5051，差 -0.0088；仍优于基线：是
白酒 Ridge：最终评估 MAE 0.6194，验证集 0.5814，差 +0.0380；仍优于基线：否
白酒 随机森林：最终评估 MAE 0.5752，验证集 0.5474，差 +0.0279；仍优于基线：是
酒类 quality  rows  feature_groups  baseline_mae  Ridge_mae  随机森林_mae
红酒       3     2               2           3.0     1.9790    2.2148
红酒       4    11              11           2.0     1.4684    1.3926
红酒       5   141             116           1.0     0.3268    0.3262
红酒       6   119             107           0.0     0.4648    0.4613
红酒       7    41              34           1.0     0.8237    0.7887
红酒       8     4               4           2.0     1.6470    1.2172
白酒       3     4               4           3.0     2.8699    2.2732
白酒       4    33              31           2.0     1.3764    1.2489
白酒       5   293             235           1.0     0.6114    0.5438
白酒       6   438             358           0.0     0.3657    0.3920
白酒       7   181             138      

In [5]:
conclusion = "；".join(f"{WINES[w]} {c} 最终评估 MAE {v['mae']:.4f}（基线 {summary[w]['baseline']['mae']:.4f}，验证集 {v['validation_mae_from_7_1']:.4f}）"
                     for w in WINES for c, v in summary[w]["candidates"].items())
run.set_conclusion(conclusion, validity="有效")
run.end()
print(conclusion)


红酒 Ridge 最终评估 MAE 0.5090（基线 0.6855，验证集 0.5194）；红酒 随机森林 最终评估 MAE 0.4963（基线 0.6855，验证集 0.5051）；白酒 Ridge 最终评估 MAE 0.6194（基线 0.6469，验证集 0.5814）；白酒 随机森林 最终评估 MAE 0.5752（基线 0.6469，验证集 0.5474）
